# 第03章　离散的世界：D组整合实验

本 Notebook 对统稿中的四节核心结论做可重复的数值检查。运行环境为 Python 3.12，
核心依赖为 NumPy、SciPy、Matplotlib 与 SymPy。执行结果由重新启动的内核顺序运行后保存。

基线：`main@c1f4d2b45178ae28bdbacfaedfdfdf8cd1a98839`。主稿和补充稿的选择见同章 `整合-正文.md` 与 D组报告。


## 3.1 ODE：Euler 与 RK4

以 $y'=-y,\ y(0)=1$ 检查两种方法的收敛阶。

In [1]:
import json
import math
import numpy as np

T = 1.0
exact_T = math.exp(-T)

def euler(h):
    y = 1.0
    n = int(round(T/h))
    for _ in range(n):
        y += h*(-y)
    return y

def rk4(h):
    y = 1.0
    t = 0.0
    n = int(round(T/h))
    for _ in range(n):
        k1 = -y
        k2 = -(y + 0.5*h*k1)
        k3 = -(y + 0.5*h*k2)
        k4 = -(y + h*k3)
        y += h*(k1 + 2*k2 + 2*k3 + k4)/6
        t += h
    return y

hs = np.array([0.2, 0.1, 0.05, 0.025])
euler_errors = np.array([abs(euler(h)-exact_T) for h in hs])
rk4_errors = np.array([abs(rk4(h)-exact_T) for h in hs])
print("Euler 误差:", euler_errors)
print("RK4 误差:", rk4_errors)
print("Euler 误差比:", euler_errors[:-1]/euler_errors[1:])
print("RK4 误差比:", rk4_errors[:-1]/rk4_errors[1:])
assert np.allclose(euler_errors[:-1]/euler_errors[1:], 2.0, rtol=0.08)
assert np.all(rk4_errors[:-1]/rk4_errors[1:] > 13)

Euler 误差: [0.04019944 0.019201   0.00939352 0.004647  ]
RK4 误差: [5.79695386e-06 3.33241056e-07 1.99760973e-08 1.22274202e-09]
Euler 误差比: [2.09361174 2.04406906 2.02141514]
RK4 误差比: [17.39567725 16.68199001 16.33713165]


## 3.2 PDE 有限差分：FTCS 稳定性

对一维热方程使用正弦初值，比较满足与违反 $r=\Delta t/\Delta x^2\le 1/2$ 时的最大幅值。

In [2]:
def ftcs_amplification(r, steps=200):
    values = np.sin(39*np.pi*np.linspace(0, 1, 41))
    interior = values[1:-1]
    for _ in range(steps):
        interior = interior + r*(np.concatenate(([0.0], interior[:-1]))
                                 - 2*interior
                                 + np.concatenate((interior[1:], [0.0])))
    return np.max(np.abs(interior))

for r in [0.49, 0.51]:
    amplitude = ftcs_amplification(r)
    print(f"r={r:.2f}, 200步后最大幅值={amplitude:.6e}")
assert ftcs_amplification(0.49) < 1.0
assert ftcs_amplification(0.51) > 1.0

r=0.49, 200步后最大幅值=1.515236e-04
r=0.51, 200步后最大幅值=1.392074e+03


## 3.3 一维有限元

对 $-u''=1,\ u(0)=u(1)=0$ 用线性有限元离散；该问题的解是二次函数，
因此线性元在节点上应当精确重现（差异只允许是舍入误差）。

In [3]:
def fem_solve(n):
    h = 1.0/n
    K = np.zeros((n+1, n+1))
    F = np.zeros(n+1)
    for element in range(n):
        local = np.array([[1.0, -1.0], [-1.0, 1.0]])/h
        indices = [element, element+1]
        K[np.ix_(indices, indices)] += local
        F[indices] += h/2.0
    free = np.arange(1, n)
    u = np.zeros(n+1)
    u[free] = np.linalg.solve(K[np.ix_(free, free)], F[free])
    x = np.linspace(0, 1, n+1)
    return x, u

x8, u8 = fem_solve(8)
error8 = np.max(np.abs(u8 - 0.5*x8*(1-x8)))
x16, u16 = fem_solve(16)
error16 = np.max(np.abs(u16 - 0.5*x16*(1-x16)))
print(f"N=8 最大节点误差 = {error8:.6e}")
print(f"N=16 最大节点误差 = {error16:.6e}")
assert error8 < 1e-12 and error16 < 1e-12

N=8 最大节点误差 = 1.387779e-17
N=16 最大节点误差 = 1.387779e-17


## 3.4 Neural ODE / PINN 的验证口径

用同一张含噪轨迹检查拟合残差与独立求解误差并不等价：插值节点上的误差可以很小，
但相邻区间仍可能偏离参考解。

In [4]:
t = np.linspace(0, 1, 21)
reference = np.exp(-t)
observed = reference + 0.04*np.sin(7*np.pi*t)
training_predictions = observed.copy()
fit_points_error = np.max(np.abs(training_predictions-observed))
dense_t = np.linspace(0, 1, 501)
dense_reference = np.exp(-dense_t)
interpolated = np.interp(dense_t, t, observed)
dense_error = np.max(np.abs(interpolated-dense_reference))
print(f"训练采样点最大偏差 = {fit_points_error:.6e}")
print(f"独立密集验证最大误差 = {dense_error:.6e}")
assert fit_points_error == 0.0 and dense_error > 0.01

训练采样点最大偏差 = 0.000000e+00
独立密集验证最大误差 = 4.000000e-02


## 执行记录

以下单元把关键结果写入同章 `数据/整合-实验摘要.json`，供网页、报告和后续复核共用。

In [5]:
summary = {
    "euler_errors": euler_errors.tolist(),
    "rk4_errors": rk4_errors.tolist(),
    "ftcs_at_049": float(ftcs_amplification(0.49)),
    "ftcs_at_051": float(ftcs_amplification(0.51)),
    "fem_error_8": float(error8),
    "fem_error_16": float(error16),
    "fit_points_error": float(fit_points_error),
    "dense_error": float(dense_error),
}
data_dir = __import__("pathlib").Path("数据")
data_dir.mkdir(exist_ok=True)
(data_dir/"整合-实验摘要.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)+"\n",
    encoding="utf-8",
)
print("整合实验摘要已保存到 数据/整合-实验摘要.json")

整合实验摘要已保存到 数据/整合-实验摘要.json


## 结论与边界

本 Notebook 验证的是统稿中的核心数值结论和可复现性；它不替代定理证明、A 组人工判断
或 B/C 的独立复核。图表与结果均来自本次重启内核后的顺序执行，没有沿用源 Notebook 输出。